# 01 — Problem Framing & Ingestion — Mall Customer Segmentation (200 loyalty members)

**Stages 0 and 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Stage 0 is a thinking step, so it
is written down here *before* any data is touched: the question, the stakeholder, the issue tree,
the SCR hypothesis and the ghost deck. Stage 1 then loads the file exactly as delivered, proves what
it is, and records what it carries *before* anyone cleans anything.

The data review (`DOCS/mall_customers_analysis.md`) was written from a first pass over the file. This
notebook re-derives each of its claims from the raw CSV rather than trusting it. One of them does not
hold: `CustomerID` is **not** free of information (§1.5), and that changes how far the file can be
used to size anything.

## 0 — Problem framing

### 0.1 The business question (one sentence)

> **Which natural segments exist among the mall's loyalty members on income and spending, which of
> them should marketing retain and which should it try to convert, and how far can this file be
> trusted to size those targets?**

The review's original framing — "who can be easily converted?" — is not answerable from this file:
there is no campaign exposure, response or transaction in it. The question above keeps the intent
(where to point the marketing budget) and adds the part the file *can* answer (who spends, who could)
and the part it must be honest about (how big each group really is).

### 0.2 Stakeholder and decision

| | |
|---|---|
| **Primary stakeholder** | Head of CRM & Marketing at the mall |
| **Decision this feeds** | How to split next quarter's loyalty-marketing budget across segments (retain / convert / maintain / deprioritise), and which conversion pilot to run |
| **What "done" looks like** | (1) A segment rule the CRM team can apply to the full member base, with evidence it is stable; (2) a named retain target and a named convert target, with profiles; (3) a pilot design with a sample size, so "convertible" can be proven rather than assumed |
| **Analysis type** | Rungs 1–2 (descriptive / diagnostic, **Path A**) with unsupervised segmentation (clustering tools from Path B, but no target variable). "Convert" is an intervention, so **Path C applies prospectively**: the deliverable is an experiment design, not a causal estimate. No causal claim is made from this file |
| **Modality & grain** | Cross-sectional tabular. **One row = one loyalty member**, n = 200, a single snapshot with no dates |

### 0.3 Scope boundaries — explicitly **out**

- Predicting campaign response, churn, CLV or visit frequency — none is in the file.
- Market basket analysis — that needs item-level transactions; segmentation is not basket analysis.
- "Why does a member spend?" — the Spending Score is an opaque mall-assigned number.
- Currency conversion (k$ → IDR) and any claim about a specific real mall's member base.

### 0.4 MECE issue tree

```
Which segments should marketing retain or convert, and how far can this file size them?
│
├── A. The file       — what is this instrument, and can its proportions be trusted?
│     A1 schema & data-review re-derivation          A2 construction signatures (order, duplicates of value)
│     A3 what the Spending Score is (and is not)
│
├── B. The structure  — are there natural segments, and are they robust?
│     B1 how many (k selection against a no-structure null)
│     B2 robustness: seeds, algorithms, bootstrap    B3 can it be written as a rule CRM can run?
│
├── C. The drivers    — what separates members, apart from the clustering inputs?
│     C1 income vs spending (level vs spread)        C2 age             C3 gender
│
└── D. The target     — whom to retain, whom to convert, and how to prove it?
      D1 segment profiles                           D2 who in the convert segment is plausibly convertible
      D3 pilot design and sample size
```

Branches are mutually exclusive by *object* (the instrument / the grouping / the attributes / the
action) and together exhaust the business question.

### 0.5 SCR — Stage 0 draft (a hypothesis, to be revised in `06`)

| | Draft |
|---|---|
| **Situation** | The mall scores 200 loyalty members on spending (1–100) and records their age, gender and annual income. |
| **Complication** | Marketing wants to know whom to target, but the file has no outcome: no campaign response, no purchases. The usual answer from this dataset — five income × spending segments — gets quoted as if segment sizes were market shares. |
| **Resolution (hypothesis)** | Five segments exist; income alone does not drive spending; age matters and gender does not. The high-income, low-spending group is the biggest conversion upside — but conversion must be proven with a test, not inferred from a clustering. |

### 0.6 Ghost deck

```
SLIDE 1 — Governing Thought   [TBD after analysis]

PART I · The file
  "This file can [/ cannot] size segments — [what the construction checks show]"
  [Placeholder: construction evidence; review claims vs file]

PART II · The structure
  "Five segments, [x]× stronger than chance, reproduced by a two-threshold rule [y]% of the time"
  [Placeholder: silhouette by k vs null band; robustness table]

PART III · The drivers
  "Income does not set how much members spend; age does [and gender does not]"
  [Placeholder: income vs score; age vs score]

PART IV · The target
  "Retain [segment]; pilot [segment, sub-group]; a credible test needs [n] members per arm"
  [Placeholder: segment map (key exhibit); age by segment; power curve]

RECOMMENDATION — rule rollout · retain · pilot · data asks
APPENDIX — cleaning log, tests, segmentation card, limits
```

### Stage 0 — Gate Checklist

- [x] Business question is a single, unambiguous sentence
- [x] Issue tree has ≥ 2 levels of decomposition and is MECE
- [x] "Done" is defined as concrete outcomes (a rule, two named targets, a pilot sample size)
- [x] Primary stakeholder named, decision context documented
- [x] Scope boundaries explicit (response, CLV, basket analysis, causal — all out)
- [x] SCR drafted · [x] Ghost deck exists with action-title placeholders · [x] Governing Thought drafted (tentative)

## 1 — Ingestion

### 1.1 Setup

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Mall_Customers.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"


def _plain(v):
    if isinstance(v, dict):
        return {str(k): _plain(x) for k, x in v.items()}
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text()) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True))
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

import hashlib, shutil
from datetime import datetime, timezone

SRC = ARCHIVE / "Mall_Customers.csv"
if KF_PATH.exists():
    KF_PATH.unlink()          # 01 starts the key-figures file afresh; 02–05 append to it
print(f"project root: {ROOT.name}")
print(f"source file : {SRC.relative_to(ROOT)}  ({SRC.stat().st_size:,} bytes)")

project root: Mall Customer Segmentation Data
source file : archive\Mall_Customers.csv  (3,981 bytes)


### 1.2 Load exactly as delivered, preserve an immutable copy, log lineage

No type coercion, no renaming, no cleaning. The SHA-256 is recorded so any later run can prove it
is reading the same bytes.

In [2]:
raw = pd.read_csv(SRC, encoding="utf-8")
sha = hashlib.sha256(SRC.read_bytes()).hexdigest()

immutable = RAW / "mall_customers_raw_immutable.csv"
shutil.copyfile(SRC, immutable)
assert hashlib.sha256(immutable.read_bytes()).hexdigest() == sha

lineage = pd.DataFrame([{
    "file": SRC.name, "source": "Kaggle 'Mall Customer Segmentation Data' (vjchoudhary7)",
    "pulled_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "rows": raw.shape[0], "cols": raw.shape[1], "bytes": SRC.stat().st_size, "sha256": sha,
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)
print(lineage.T.to_string(header=False))
print()
print(raw.dtypes.to_string())
raw.head(6)

file                                                      Mall_Customers.csv
source               Kaggle 'Mall Customer Segmentation Data' (vjchoudhary7)
pulled_utc                                                  2026-10-14 09:50
rows                                                                     200
cols                                                                       5
bytes                                                                   3981
sha256      416a4f62a8f33841a16e1db5297f0e05d3b65b05e73632cedae58e739e1e1f77

CustomerID                int64
Gender                      str
Age                       int64
Annual Income (k$)        int64
Spending Score (1-100)    int64


,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
5,6,Female,22,17,76


### 1.3 Schema contract

A written contract of what each column must be. Every check is run; failures are recorded, not
silently fixed.

In [3]:
EXPECTED = ["CustomerID", "Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"]
INC, SCORE = "Annual Income (k$)", "Spending Score (1-100)"

contract = [
    ("columns exactly as documented", list(raw.columns) == EXPECTED, f"{raw.shape[1]} columns"),
    ("no missing values", raw.notna().all().all(), f"{int(raw.isna().sum().sum())} nulls"),
    ("CustomerID unique", raw["CustomerID"].is_unique, f"{raw['CustomerID'].nunique()} distinct"),
    ("CustomerID contiguous 1..n", set(raw["CustomerID"]) == set(range(1, len(raw) + 1)), "1–200"),
    ("Gender in {Male, Female}", set(raw["Gender"]) <= {"Male", "Female"}, str(sorted(raw["Gender"].unique()))),
    ("Age integer in 18–100", raw["Age"].between(18, 100).all() and raw["Age"].dtype.kind == "i",
     f"{raw['Age'].min()}–{raw['Age'].max()}"),
    ("Income integer > 0", (raw[INC] > 0).all() and raw[INC].dtype.kind == "i",
     f"{raw[INC].min()}–{raw[INC].max()} k$"),
    ("Spending Score integer in 1–100", raw[SCORE].between(1, 100).all() and raw[SCORE].dtype.kind == "i",
     f"{raw[SCORE].min()}–{raw[SCORE].max()}"),
    ("no duplicate members (excl. ID)", not raw.drop(columns="CustomerID").duplicated().any(),
     f"{int(raw.drop(columns='CustomerID').duplicated().sum())} duplicates"),
    ("CustomerID carries no information (review §1)",
     abs(raw["CustomerID"].corr(raw[INC], method="spearman")) < 0.2,
     f"Spearman ρ(ID, income) = {raw['CustomerID'].corr(raw[INC], method='spearman'):.3f}"),
]
schema = pd.DataFrame(contract, columns=["check", "pass", "detail"])
schema.to_csv(RAW / "_schema_validation.csv", index=False)
print(f"{schema['pass'].sum()} of {len(schema)} contract checks pass\n")
schema

9 of 10 contract checks pass



,check,pass,detail
0,columns exactly as documented,True,5 columns
1,no missing values,True,0 nulls
2,CustomerID unique,True,200 distinct
3,CustomerID contiguous 1..n,True,1–200
4,"Gender in {Male, Female}",True,"['Female', 'Male']"
5,Age integer in 18–100,True,18–70
6,Income integer > 0,True,15–137 k$
7,Spending Score integer in 1–100,True,1–99
8,no duplicate members (excl. ID),True,0 duplicates
9,CustomerID carries no information (review §1),False,"Spearman ρ(ID, income) = 1.000"


Nine of ten checks pass: the file is complete, typed and in range. The one failure is the review's
own claim that `CustomerID` is "unique, sequential, no information content". It is unique and
sequential, but its rank correlation with income is close to 1. §1.5 shows why.

### 1.4 Re-deriving the data review's headline figures

In [4]:
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

a, inc, sc, g = raw["Age"], raw[INC], raw[SCORE], raw["Gender"]
r = raw[["Age", INC, SCORE]].corr()
X = StandardScaler().fit_transform(raw[[INC, SCORE]])
sil = {k: silhouette_score(X, KMeans(k, n_init=10, random_state=RANDOM_STATE).fit_predict(X))
       for k in range(2, 9)}
bands = pd.cut(a, [17, 25, 35, 45, 55, 70], labels=["18–25", "26–35", "36–45", "46–55", "56–70"])
band_means = sc.groupby(bands, observed=True).agg(["size", "mean"]).round(1)

checks = pd.DataFrame([
    ("shape", "200 × 5", f"{raw.shape[0]} × {raw.shape[1]}"),
    ("female / male", "112 (56%) / 88 (44%)", f"{(g=='Female').sum()} ({(g=='Female').mean():.0%}) / {(g=='Male').sum()} ({(g=='Male').mean():.0%})"),
    ("age range / mean / median", "18–70 / 38.8 / 36", f"{a.min()}–{a.max()} / {a.mean():.1f} / {a.median():.0f}"),
    ("income range / mean / median", "15–137 / 60.6 / 61.5", f"{inc.min()}–{inc.max()} / {inc.mean():.1f} / {inc.median():.1f}"),
    ("score range / mean / median", "1–99 / 50.2 / 50", f"{sc.min()}–{sc.max()} / {sc.mean():.1f} / {sc.median():.0f}"),
    ("customers with income > 100k", "14", f"{(inc > 100).sum()}"),
    ("skew age / income / score", "0.49 / 0.32 / −0.05", f"{a.skew():.2f} / {inc.skew():.2f} / {sc.skew():.2f}"),
    ("r age–score / age–income / income–score", "−0.33 / −0.01 / 0.01",
     f"{r.loc['Age', SCORE]:.2f} / {r.loc['Age', INC]:.2f} / {r.loc[INC, SCORE]:.2f}"),
    ("mean score female / male", "51.5 / 48.5", f"{sc[g=='Female'].mean():.1f} / {sc[g=='Male'].mean():.1f}"),
    ("silhouette peak (k, value)", "k = 5, 0.555", f"k = {max(sil, key=sil.get)}, {max(sil.values()):.3f}"),
    ("mean score by age band", "54.9 / 64.4 / 44.8 / 36.7 / 38.4",
     " / ".join(f"{v:.1f}" for v in band_means["mean"])),
    ("age-band sizes", "38 / 60 / 36 / 37 / 29", " / ".join(str(int(v)) for v in band_means["size"])),
], columns=["figure", "review says", "file says"])
checks

,figure,review says,file says
0,shape,200 × 5,200 × 5
1,female / male,112 (56%) / 88 (44%),112 (56%) / 88 (44%)
2,age range / mean / median,18–70 / 38.8 / 36,18–70 / 38.9 / 36
3,income range / mean / median,15–137 / 60.6 / 61.5,15–137 / 60.6 / 61.5
4,score range / mean / median,1–99 / 50.2 / 50,1–99 / 50.2 / 50
5,customers with income > 100k,14,14
6,skew age / income / score,0.49 / 0.32 / −0.05,0.49 / 0.32 / -0.05
7,r age–score / age–income / income–score,−0.33 / −0.01 / 0.01,-0.33 / -0.01 / 0.01
8,mean score female / male,51.5 / 48.5,51.5 / 48.5
9,"silhouette peak (k, value)","k = 5, 0.555","k = 5, 0.555"


Every headline figure in the review reproduces (mean age is 38.85, which the review rounds to 38.8).
The problem with the review is not its arithmetic. It is one sentence in its data dictionary.

### 1.5 Construction audit — `CustomerID` is the income rank, and the file is built in pairs

In [5]:
ids = raw["CustomerID"].to_numpy()
inc_v, sc_v = raw[INC].to_numpy(), raw[SCORE].to_numpy()

sorted_by_income = bool(raw.sort_values("CustomerID")[INC].is_monotonic_increasing)
pairs_equal = inc_v[0::2] == inc_v[1::2]              # rows (1,2), (3,4), … share an income?
mult = raw[INC].value_counts().value_counts().sort_index()

print(f"income non-decreasing in CustomerID order : {sorted_by_income}")
print(f"distinct income values                    : {raw[INC].nunique()} for {len(raw)} members")
print(f"how often each income value occurs        : {dict(mult)}   (every count is even)")
print(f"consecutive ID pairs (1,2),(3,4),… with the same income: {pairs_equal.sum()} of {len(pairs_equal)}")

income non-decreasing in CustomerID order : True
distinct income values                    : 64 for 200 members
how often each income value occurs        : {2: np.int64(42), 4: np.int64(14), 6: np.int64(6), 12: np.int64(2)}   (every count is even)
consecutive ID pairs (1,2),(3,4),… with the same income: 100 of 100


Every income value appears an **even** number of times, and all 100 consecutive ID pairs share an
income. The rows were created two at a time and sorted by income before the IDs were assigned. That
alone is harmless. What each twin's Spending Score looks like is not:

In [6]:
pair = pd.DataFrame({"income": inc_v[0::2], "odd_score": sc_v[0::2], "even_score": sc_v[1::2]})
pair["band"] = pd.cut(pair.income, [0, 40, 69, 200], labels=["≤ 40k", "41–69k", "≥ 70k"])
summary = pair.groupby("band", observed=True).agg(
    pairs=("income", "size"),
    odd_twin_mean=("odd_score", "mean"), odd_twin_max=("odd_score", "max"),
    even_twin_mean=("even_score", "mean"), even_twin_min=("even_score", "min"),
    even_higher=("income", lambda s: int((pair.loc[s.index, "even_score"] > pair.loc[s.index, "odd_score"]).sum())),
).round(1)
summary

,pairs,odd_twin_mean,odd_twin_max,even_twin_mean,even_twin_min,even_higher
band,,,,,,
≤ 40k,25,23.1,55,75.8,42,23
41–69k,37,50.5,60,50.7,40,19
≥ 70k,38,18.6,40,81.9,63,38


At both income extremes the pair is split by ID parity: the **odd-ID twin is a low spender and the
even-ID twin a high spender**, without exception above 70k. In the middle band both twins sit near 50.
This is a generator signature: the "four corner" segments every tutorial finds in this file were
filled one member from each pair, so **they come out roughly equal in size by construction**.

What this does and does not mean:

| | |
|---|---|
| **Does not mean** | The segments are fake for analysis purposes. Their shape (who is in which corner, what thresholds separate them) is still the structure this file contains, and `05` tests how robust it is |
| **Does mean** | Segment **sizes** in this file are not market shares. "17.5% of members are Under-engaged Affluent" is a property of the generator, not of a mall. Any budget sized on these proportions is sized on an artifact |
| **Handling** | `CustomerID` (and anything derived from it) is excluded from every feature set; its parity is kept only as a forensic column so `05` can show the link to segments |

In [7]:
extremes = pair[pair.band != "41–69k"]
fisher = stats.fisher_exact([[int((extremes.even_score > 50).sum()), int((extremes.even_score <= 50).sum())],
                             [int((extremes.odd_score > 50).sum()), int((extremes.odd_score <= 50).sum())]])
print(f"pairs at the income extremes (≤40k or ≥70k): {len(extremes)}")
print(f"  even twin scores > 50: {(extremes.even_score > 50).sum()} of {len(extremes)}")
print(f"  odd twin scores  > 50: {(extremes.odd_score > 50).sum()} of {len(extremes)}")
print(f"  Fisher exact p = {fisher.pvalue:.1e}")

pairs at the income extremes (≤40k or ≥70k): 63
  even twin scores > 50: 61 of 63
  odd twin scores  > 50: 1 of 63
  Fisher exact p = 4.1e-32


### 1.6 What the Spending Score is — and is not

The review is right that the score is mall-assigned and undefined. Two more facts from the file:

- It is spread almost evenly over 1–99 (skew −0.05). Behavioural spend measures (basket value, visit
  counts) are almost always right-skewed, so this is a **rank-like score**, not currency.
- It is **not an outcome**. Nothing in the file records a member responding to anything. A model
  trained to predict the score would only reproduce the mall's own rule.

So the score can describe *who spends now*. It cannot say *who would respond to a campaign*. That
gap is why `05` ends with a pilot design.

### 1.7 Governance

In [8]:
gov = pd.DataFrame([
    ("PII present", "pseudonymous ID only; no names or contact data"),
    ("personal attributes", "age and gender: targeting on them needs a documented, lawful basis"),
    ("licence", "public Kaggle teaching dataset; portfolio use"),
    ("measurement", "Spending Score is mall-assigned and undefined; income meaning (individual or household) undocumented"),
    ("construction", "rows generated in income-sorted pairs; segment sizes are not market shares"),
    ("fairness audit", "segment × gender and target-rule × age checked in 05"),
], columns=["item", "assessment"])
gov.to_csv(RAW / "_governance.csv", index=False)
gov

,item,assessment
0,PII present,pseudonymous ID only; no names or contact data
1,personal attributes,age and gender: targeting on them needs a docu...
2,licence,public Kaggle teaching dataset; portfolio use
3,measurement,Spending Score is mall-assigned and undefined;...
4,construction,rows generated in income-sorted pairs; segment...
5,fairness audit,segment × gender and target-rule × age checked...


In [9]:
kf_update(
    n_rows_raw=len(raw), n_cols_raw=raw.shape[1], sha256=sha,
    n_female=int((g == "Female").sum()), n_male=int((g == "Male").sum()),
    age_min=int(a.min()), age_max=int(a.max()), age_mean=float(a.mean()), age_median=float(a.median()),
    income_min=int(inc.min()), income_max=int(inc.max()), income_mean=float(inc.mean()),
    income_median=float(inc.median()),
    score_mean=float(sc.mean()), score_median=float(sc.median()), score_skew=float(sc.skew()),
    n_income_over_100=int((inc > 100).sum()),
    n_schema_checks=len(schema), n_schema_pass=int(schema["pass"].sum()),
    id_income_spearman=float(raw["CustomerID"].corr(raw[INC], method="spearman")),
    n_income_values=int(raw[INC].nunique()), n_pairs=int(len(pairs_equal)),
    n_pairs_same_income=int(pairs_equal.sum()),
    n_extreme_pairs=int(len(extremes)),
    n_extreme_even_high=int((extremes.even_score > 50).sum()),
    n_extreme_odd_high=int((extremes.odd_score > 50).sum()),
    pair_fisher_p=float(fisher.pvalue),
    silhouette_review_k5=float(sil[5]),
)

  _key_figures.json <- n_rows_raw, n_cols_raw, sha256, n_female, n_male, age_min, age_max, age_mean, age_median, income_min, income_max, income_mean, income_median, score_mean, score_median, score_skew, n_income_over_100, n_schema_checks, n_schema_pass, id_income_spearman, n_income_values, n_pairs, n_pairs_same_income, n_extreme_pairs, n_extreme_even_high, n_extreme_odd_high, pair_fisher_p, silhouette_review_k5  (28 figures total)


## Stage 1 — Gate Checklist

- [x] Raw data saved to `data/raw/mall_customers_raw_immutable.csv`, hash-verified
- [x] Source, timestamp, row/column counts and SHA-256 logged (`data/raw/_lineage.csv`)
- [x] Schema checked against a written contract; the one failure (ID carries income) documented, not fixed
- [x] Governance review recorded (`data/raw/_governance.csv`): pseudonymous, two personal attributes
- [x] Data review re-derived from the raw file: every figure reproduces; **one claim does not** (ID
      "has no information content") and it is corrected here

**Carried into `02`:** exclude `CustomerID` from features · keep ID parity as a forensic flag only ·
no rows to drop · derived columns from review §7 · treat segment sizes as non-transferable.